In [15]:
import pickle
import pandas as pd 
import networkx as nx 

In [16]:
merged = pd.read_csv(r"C:\Users\MR.CO\Desktop\London Uderground\Cleaned_datasets\merged.csv")
stations = pd.read_csv(r"C:\Users\MR.CO\Desktop\London Uderground\Cleaned_datasets\stations_c.csv")

In [17]:
merged.head()

,Line,Direction,Station from (A),Station to (B),Distance (Kms),from_id,to_id
0,Bakerloo,Southbound,harrow & wealdstone,kenton,1.74,114,140
1,Bakerloo,Southbound,kenton,south kenton,1.40,140,237
2,Bakerloo,Southbound,south kenton,north wembley,0.90,237,185
3,Bakerloo,Southbound,north wembley,wembley central,1.27,185,281
4,Bakerloo,Southbound,wembley central,stonebridge park,1.71,281,246


In [18]:
edges = merged.copy()
# storing ids in "stations_a" and "stations_b" in ascending
edges["station_a"] = edges[["from_id","to_id"]].min(axis = 1)
edges["station_b"] = edges[["from_id","to_id"]].max(axis = 1)

edges = edges.groupby(["station_a","station_b"],as_index=False).agg(
    {"Distance (Kms)": "min",
     "Line" : "first"})

In [19]:
edges.head()

,station_a,station_b,Distance (Kms),Line
0,1,52,1.21,District
1,1,73,1.03,Piccadilly
2,1,110,4.41,Piccadilly
3,1,234,2.20,Piccadilly
4,2,156,0.61,Metropolitan


In [20]:
# sanity check for no more duplicate station pairs
len(edges) == len(edges.groupby(["station_a","station_b"]))

True

In [21]:
idx_to_drop = stations[stations["name"] == "all saints"].index.to_list()
stations.drop(idx_to_drop,axis = 0,inplace = True)

In [22]:

id_to_name = stations.set_index("id")["name"].to_dict()
name_to_id = {v: k for k, v in id_to_name.items()}

In [23]:
G = nx.Graph()

# Iterates through every row and adds each station route as an edges (station_a to station_b with their distance and Line as one edge)
for _, row in edges.iterrows(): 
    G.add_edge(row["station_a"],row["station_b"],weight = row["Distance (Kms)"], line = row["Line"])

In [24]:
print(G.get_edge_data(1,73))

{'weight': 1.03, 'line': 'Piccadilly'}


In [25]:
# Testing the graph 
station_a = 1
station_b = 73

distance_km = nx.dijkstra_path_length(G,source = station_a, target = station_b, weight = "weight")
path = nx.dijkstra_path(G, source = station_a, target = station_b, weight = "weight")

In [26]:
print(f"{distance_km:.2f} km")

1.03 km


In [27]:
path

[1, 73]

In [28]:
#saving the graph and dictionaries 
with open("graph.pkl", "wb") as f:
    pickle.dump(G, f)

with open("id_to_name.pkl", "wb") as f:
    pickle.dump(id_to_name, f)

with open("name_to_id.pkl", "wb") as f:
    pickle.dump(name_to_id, f)